# Predictive analysis of naval incidents in the USA, 2002 - 2015: <br>
## Model deployment: Server examples for VesselBalancedSample

> Author: [Oscar Anton](https://www.linkedin.com/in/oscanton/) <br>
> Date: 2024 <br>
> License: [CC BY-NC-ND 4.0 DEED](https://creativecommons.org/licenses/by-nc-nd/4.0/) <br>
> Version: 0.9 <br>

# 0. Loadings

### Libraries

In [1]:
import pandas as pd
import joblib

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn

from typing import List
from fastapi.middleware.cors import CORSMiddleware

import threading

### Model

In [2]:
# Load model
models_path = '../5.DataModel/Models/'
rf_model = joblib.load(models_path + 'rf_train.pkl')

# Check parameters
params = rf_model.get_params()
for param_name, param_value in params.items():
    print(f" {param_name}: {param_value}")

 bootstrap: True
 ccp_alpha: 0.0
 class_weight: None
 criterion: gini
 max_depth: None
 max_features: sqrt
 max_leaf_nodes: None
 max_samples: None
 min_impurity_decrease: 0.0
 min_samples_leaf: 1
 min_samples_split: 2
 min_weight_fraction_leaf: 0.0
 monotonic_cst: None
 n_estimators: 100
 n_jobs: None
 oob_score: False
 random_state: 42
 verbose: 0
 warm_start: False


### Scaler

In [3]:
# Load used scaler for the trainings
datasets_path = '../5.DataModel/Datasets/'
scaler = joblib.load(datasets_path + 'scaler.pkl')

# Check scalable variables
print(f'Scalable variables: {scaler.feature_names_in_}')

['gross_ton' 'vessel_length']


# 1. Variable transformation & encoding

### Imput data validation

In [4]:
# Define expected columns entered by the user
expected_columns = [
    'vessel_class', 'build_year', 'flag_abbr',
    'classification_society', 'solas_desc',
    'gross_ton', 'vessel_length'
]

# Columns and data types comparations
def input_validation(df):
    # Check if all expected columns are present
    missing_columns = set(expected_columns) - set(df.columns)
    if missing_columns:
        raise ValueError(f"Missing expected columns: {missing_columns}")
    
    # Check if there are any extra columns
    extra_columns = set(df.columns) - set(expected_columns)
    if extra_columns:
        raise ValueError(f"Unexpected columns: {extra_columns}")
    
    # Check data types
    expected_dtypes = {
        'vessel_class': object, # object type in pandas is equivalent to str in Python
        'build_year': 'int64',
        'flag_abbr': object,
        'classification_society': object,
        'solas_desc': object,
        'gross_ton': 'float64', # Using float64 to allow both int and float
        'vessel_length': 'float64' # Using float64 to allow both int and float
    }
    
    for column, dtype in expected_dtypes.items():
        if not pd.api.types.is_dtype_equal(df[column].dtype, dtype):
            raise ValueError(f"Column '{column}' must be of type {dtype}")
    
    return True

### Input data formating

In [5]:
# Define expected variables by the model
input_columns = [
    'vessel_class_Barge', 'vessel_class_Bulk Carrier',
    'vessel_class_Fishing Vessel', 'vessel_class_General Dry Cargo Ship',
    'vessel_class_Miscellaneous Vessel', 'vessel_class_Offshore',
    'vessel_class_Passenger Ship', 'vessel_class_Recreational',
    'vessel_class_Tank Ship', 'vessel_class_Towing Vessel',
    'vessel_class_other value', 'build_year_very Old', 'build_year_old',
    'build_year_average', 'build_year_new', 'build_year_very new',
    'flag_abbr_CA', 'flag_abbr_LR', 'flag_abbr_PA', 'flag_abbr_US',
    'flag_abbr_other value',
    'classification_society_AMERICAN BUREAU OF SHIPPING',
    'classification_society_DET NORSKE VERITAS',
    "classification_society_LLOYD'S REGISTER OF SHIPPING",
    'classification_society_NIPPON KAIJI KYOKAI',
    'classification_society_UNSPECIFIED',
    'classification_society_other value', 'solas_desc_Active SOLAS',
    'solas_desc_Historical SOLAS', 'solas_desc_Non SOLAS', 'gross_ton',
    'vessel_length'
]

# Transform categorical variables to one hot codifing
def value_ohe(variable_name, values):
    one_hot_df = []
    labels = list(filter(lambda x: x.startswith(variable_name), input_columns))
    for value in values:
        target_column = variable_name + '_' + value
        one_hot_row = pd.DataFrame(0, index=[0], columns=labels)
        one_hot_row[target_column] = 1
        one_hot_df.append(one_hot_row)
    # Concatenate all one-hot encoded rows into a single DataFrame
    return pd.concat(one_hot_df, ignore_index=True)

# Cutting & codifing build_year
def build_year_ohe(build_years):
    labels = ['very Old', 'old', 'average', 'new', 'very new']
    year_categories = pd.cut(build_years,
                             bins=[-float('inf'), 1940, 1960, 1980, 2000, float('inf')],
                             labels=labels,
                             include_lowest=True)
    # Return one-hot-encoded values for cutted value
    return value_ohe('build_year', year_categories)

# Transform numeric variables to scaled values
def scale_values(scaler, feature_names, **kwargs):
    # Create a DataFrame with the new entries
    new_data = pd.DataFrame(kwargs)

    # Transform the data with the scaler adjusted
    scaled_values = pd.DataFrame(scaler.transform(new_data), columns=feature_names)
    
    return scaled_values


# Define structure of data input
def input_structure(df):
    data_input = pd.concat([
        value_ohe('vessel_class', df['vessel_class']),
        build_year_ohe(df['build_year']),
        value_ohe('flag_abbr', df['flag_abbr']),
        value_ohe('classification_society', df['classification_society']),
        value_ohe('solas_desc', df['solas_desc']),
        scale_values(scaler,
                     feature_names = ['gross_ton', 'vessel_length'],
                     gross_ton = df['gross_ton'],
                     vessel_length = df['vessel_length']
                     )
        ], axis=1)
    return data_input

# 2. Predictions for new data examples

In [6]:
# Example values (for checking purposes, example = X_test.iloc[[15]] & X_test.iloc[[9075]])
input_df = pd.DataFrame({
    "vessel_class": ["Barge", "Fishing Vessel"],
    "build_year": [2005, 1970],
    "flag_abbr": ["US", "US"],
    "classification_society": ["UNSPECIFIED", "UNSPECIFIED"],
    "solas_desc": ["Non SOLAS", "Historical SOLAS"],
    "gross_ton": [764.0, 2749.0],
    "vessel_length": [200.0, 252.3]
    })

input_df

,vessel_class,build_year,flag_abbr,classification_society,solas_desc,gross_ton,vessel_length
0,Barge,2005,US,UNSPECIFIED,Non SOLAS,764.0,200.0
1,Fishing Vessel,1970,US,UNSPECIFIED,Historical SOLAS,2749.0,252.3


In [7]:
def get_prediction_probabilities(input_df):
    result = {}
    try:
        # Verify data format
        is_valid = input_validation(input_df)
        
        if not is_valid:
            raise ValueError("Invalid input data format")
        
        # Calculate probability array
        prediction_proba = rf_model.predict_proba(input_structure(input_df))
        
        # Store probability for each vessel in a dictionary
        for i, value in enumerate(prediction_proba[:, 1], start=1):
            result[f'vessel {i}'] = f'{value:.4%}'
        
    except ValueError as e:
        # Store error message in the result dictionary
        result["error"] = f"Data is invalid: {e}"
    
    return result

In [8]:
get_prediction_probabilities(input_df)

{'vessel 1': '76.9789%', 'vessel 2': '100.0000%'}

# 3. FastAPI POST requests

### Vessel incident involvement prediction

Uvicorn server running in 127.0.0.1:8000/predict

In [9]:
# Initiate FastAPI
app = FastAPI()

# Configure CORS to allow requests from any source
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Allows all origins. Change this in production.
    allow_credentials=True,
    allow_methods=["*"],  # Allows all HTTP methods
    allow_headers=["*"],  # Allow all headers
)

# Define the structure of the request data using Pydantic
class PredictionRequest(BaseModel):
    vessel_class: List[str]
    build_year: List[int]
    flag_abbr: List[str]
    classification_society: List[str]
    solas_desc: List[str]
    gross_ton: List[float]
    vessel_length: List[float]


@app.post('/predict')
def deploy_model(request: PredictionRequest):
    try:
        # Convert the request data to a DataFrame
        input_df = pd.DataFrame({
            "vessel_class": request.vessel_class,
            "build_year": request.build_year,
            "flag_abbr": request.flag_abbr,
            "classification_society": request.classification_society,
            "solas_desc": request.solas_desc,
            "gross_ton": request.gross_ton,
            "vessel_length": request.vessel_length
        })
        
        # Call the prediction function
        probabilities = get_prediction_probabilities(input_df)
        
        # Return the prediction probabilities
        return probabilities
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Define a function to run the Uvicorn server
def run_uvicorn():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

# Start the server in a separate thread
if __name__ == "__main__":
    thread = threading.Thread(target=run_uvicorn)
    thread.start()


Checking available at: <br>
https://web.postman.co/workspace/My-Workspace~dcdd54f9-b9a9-48b6-8696-4024e746d3b6/request/37058628-68105f7e-fa3d-4125-9e23-02e6d272a2af?action=share&source=copy-link&creator=37058628